<a href="https://colab.research.google.com/github/AndreaHortaHernandez/poster2movie/blob/kenya-mejoras-base/notebooks/02_resena.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Poster2Movie — 02. Reseña: cartel → sinopsis

Versión **base**: Qwen3-VL-2B-Instruct sin entrenar, con una instrucción (detalles en `p2m/resena.py`).

**Entorno: GPU (T4).**

## Preparar el entorno

Monta tu Google Drive (ahí vive el dataset, `poster2movie_datos/`), clona el repo e instala lo necesario. Si el dataset te lo compartieron, agrega la carpeta `poster2movie_datos` a *Mi unidad* como acceso directo.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["P2M_DATOS"] = "/content/drive/MyDrive/poster2movie_datos"
os.environ["P2M_SALIDAS"] = "/content/drive/MyDrive/poster2movie_kenya"   # ← descomentada, con tu carpeta
SALIDAS = os.environ.get("P2M_SALIDAS", os.environ["P2M_DATOS"])
REPO = "AndreaHortaHernandez/poster2movie"
if not os.path.isdir("/content/repo"):
    !git clone -b kenya-mejoras-base https://github.com/{REPO}.git /content/repo   # ← con -b y tu rama
else:
    !cd /content/repo && git checkout kenya-mejoras-base && git pull                # ← para que no use main
%cd /content/repo
!pip install -q -r requirements.txt


## 1. Probar con un cartel

In [ ]:
from google.colab import files
from PIL import Image
from p2m.resena import Resenador

resenador = Resenador()
for nombre in files.upload():
    print(resenador.generar(Image.open(nombre).convert('RGB')))

## 2. Evaluar en test (200 carteles, ~15-20 min)

ROUGE-L y BERTScore contra la sinopsis real. Con `--generos predichos` usa los géneros del clasificador en vez de los reales. Resultados en Drive: `poster2movie_datos/resultados/resena/` (incluye los textos generados).

In [ ]:
del resenador  # libera la GPU
!python -m p2m.evaluar_resena --cuantos 200

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 300)
pd.read_csv(SALIDAS + "/resultados/resena/base_ejemplos.csv")[["titulo", "generada", "real", "BERTScore"]].head(10)